# Random Forest and XGBoost on Descriptors & Fragment Features (Representation A)

This notebook uses features set A (refer to notebook 1a) and selects hyperparameters and models using k-fold cross-validation

**Goal:** Select the best performing model/hyperparameters through k fold cross validations using the chemical features extracted in the previous notebook.


### Import Required Libraries

- **pandas/numpy**: Data manipulation
- **seaborn/matplotlib**: Data visualization
- **sklearn**: Machine learning tools (models, cross-validation, metrics)
- **ydata_profiling**: Data profiling and analysis


In [1]:
import numpy as np
import pandas as pd
from scipy import stats

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier


#ignore warnings
import warnings
warnings.filterwarnings('ignore')

## Step 1: Load Feature Data

Load the training features extracted in the previous notebook (`training_features.csv`).

This dataset contains ~163 feature columns plus the target variable (ACTIVE).


In [2]:
features_df = pd.read_csv("training_features.csv")
features_df.head()


,Unnamed: 0,INDEX,SMILES,ACTIVE,MolFromSmiles,NumAtoms,NumHeavyAtoms,NumBonds,fr_Al_COO,fr_Al_OH,...,CalcNumRings,CalcNumRotatableBonds,CalcNumSaturatedCarbocycles,CalcNumSaturatedHeterocycles,CalcNumSaturatedRings,CalcNumSpiroAtoms,CalcNumUnspecifiedAtomStereoCenters,CalcPhi,CalcTPSA,_CalcMolWt
0,0,1,O=C(Nc1ccc2c(c1)OCCO2)C1CCN(c2ncccn2)CC1,0.0,<rdkit.Chem.rdchem.Mol object at 0x12dc3b220>,25,25,28,0,0,...,4,3,0,1,1,0,0,4.368063,76.58,340.383
1,1,2,COCCCN1C(=O)C2C(C(=O)Nc3cccc(Cl)c3)C3C=CC2(O3)...,0.0,<rdkit.Chem.rdchem.Mol object at 0x12dc38b30>,35,35,39,0,0,...,5,8,1,2,3,1,5,6.877647,96.97,502.011
2,2,3,CCSc1ncc(Cl)c(C(=O)Nc2ccccc2C)n1,0.0,<rdkit.Chem.rdchem.Mol object at 0x12dc3b0d0>,20,20,21,0,0,...,2,4,0,0,0,0,0,4.977937,54.88,307.806
3,3,4,COc1ccc2cc(/C=N/NC(=O)CN(c3ccccc3C)S(=O)(=O)c3...,0.0,<rdkit.Chem.rdchem.Mol object at 0x12dc3b140>,36,36,39,0,0,...,4,8,0,0,0,0,0,7.516779,100.96,523.014
4,4,5,CCCC(=O)Nc1nc2ccc(NC(=O)c3c(F)c(F)c(OC)c(F)c3F...,0.0,<rdkit.Chem.rdchem.Mol object at 0x12dc3b1b0>,30,30,32,0,0,...,3,6,0,0,0,0,0,6.202841,80.32,441.406


## Step 2 - Split into X (features) and y (targets)

**Separate features (X) from target (y)**: 
   - X: All feature columns (excluding ACTIVE, SMILES, INDEX, MolFromSmiles)
   - y: ACTIVE column (0 = inactive, 1 = active)

In [3]:
features_df.columns

Index(['Unnamed: 0', 'INDEX', 'SMILES', 'ACTIVE', 'MolFromSmiles', 'NumAtoms',
       'NumHeavyAtoms', 'NumBonds', 'fr_Al_COO', 'fr_Al_OH',
       ...
       'CalcNumRings', 'CalcNumRotatableBonds', 'CalcNumSaturatedCarbocycles',
       'CalcNumSaturatedHeterocycles', 'CalcNumSaturatedRings',
       'CalcNumSpiroAtoms', 'CalcNumUnspecifiedAtomStereoCenters', 'CalcPhi',
       'CalcTPSA', '_CalcMolWt'],
      dtype='object', length=164)

## Step 3: Train Machine Learning Model

Train a **Random Forest Classifier** to predict molecular activity.

**Process:**

1. **10-Fold Cross-Validation**: 
   - Split data into 10 folds - use StratifiedKFold to distribute classes better
   - Train on 9 folds, test on 1 fold
   - Repeat 10 times with different splits
   - This gives a robust estimate of model performance

2. **XGBoost Classifier**
   - Select hyperparameters to experiment on
   - Apply to k-folds
   - Good for tabular data

3. **Random Forest Classifier**:
   - Ensemble method that combines multiple decision trees
   - Good baseline model for structured data
   - Handles many features well



In [4]:
TARGET_COL = "ACTIVE"  
y = features_df[TARGET_COL]

In [5]:
non_feature_cols = [TARGET_COL, "SMILES", "MolFromSmiles", "Unnamed: 0", "INDEX"]
non_feature_cols = [c for c in non_feature_cols if c in features_df.columns]


y = features_df[TARGET_COL]

X = features_df.drop(columns=non_feature_cols)
X = X.select_dtypes(include=[np.number])

X.shape, y.shape

((202895, 159), (202895,))

Set random seed for reproducibility. This ensures that results are consistent across runs.


In [6]:
seed = 20231124
np.random.seed(seed)

A comparison between XGBoost and Random Forest
- we create a helper function to get a confidence interval in the resulting models

In [7]:
def calculate_ci(scores, confidence=0.95):
    n = len(scores)
    mean = np.mean(scores)
    std_err = stats.sem(scores)  # Standard error of the mean
    ci = stats.t.interval(confidence, n-1, loc=mean, scale=std_err)
    return mean, ci

# Divide dataset into training and test set
- training test will be split into training and validation sets (k-fold cross-validation)
- test set will be used to measure performance of the models trained during this configuration

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,  # 80% train, 20% test
    stratify=y,      # maintain class balance
    random_state=seed
)

print(f"Training set: X={X_train.shape}, y={y_train.shape}")
print(f"Test set: X={X_test.shape}, y={y_test.shape}")


Training set: X=(162316, 159), y=(162316,)
Test set: X=(40579, 159), y=(40579,)


## 3a - XGBoost hyperparam exploration on Kfolds

In [9]:
## XGB things
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
param_dist_xgb = {
    "n_estimators": [200, 300, 400, 600],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5]
}

xgb_base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1   
)

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist_xgb,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)


xgb_search.fit(X_train, y_train)


Fitting 5 folds for each of 20 candidates, totalling 100 fits


KeyboardInterrupt: 

In [ ]:

best_xgb = xgb_search.best_estimator_

auc_scores = cross_val_score(best_xgb,X,y,cv=cv,scoring="roc_auc",n_jobs=-1)

auc_scores_base = cross_val_score(xgb_base,X,y,cv=cv,scoring="roc_auc",n_jobs=-1  
)

print("Mean AUC (BASE):", auc_scores_base.mean())
print("Best model stats: ")
print("AUC per fold (best):", auc_scores)
print("Mean AUC:", auc_scores.mean())
print("Std AUC:", auc_scores.std())


print("Best XGBoost AUC:", xgb_search.best_score_)
print("Best XGBoost params:", xgb_search.best_params_)

Mean AUC (BASE): 0.8898640062311927
Best model stats: 
AUC per fold (best): [0.90095503 0.90405192 0.90458244 0.89361147 0.8941196 ]
Mean AUC: 0.899464092137085
Std AUC: 0.004738888820771231
Best XGBoost AUC: 0.899464092137085
Best XGBoost params: {'subsample': 0.8, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 0.6}


# 3b - Random Forest Hyperparams exploration on kfolds 

In [ ]:
best_base = RandomForestClassifier(
    n_jobs=-1,
    random_state=42
)

param_dist_rf = {
    # "n_estimators": [100, 200],
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [10, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
    # "bootstrap": [True]
}

best_search = RandomizedSearchCV(
    estimator=best_base,
    param_distributions=param_dist_rf,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)
best_search.fit(X_train, y_train)

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [10, 30, ...], 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,n_iter,20
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [ ]:


best_rf = best_search.best_estimator_
auc_scores_rf = cross_val_score(best_rf, X, y, cv=cv, scoring="roc_auc")
auc_scores_base_rf = cross_val_score(best_base, X, y, cv=cv, scoring="roc_auc")

print("Mean AUC (BASE):", auc_scores_base_rf.mean())
print("Best model stats: ")
print("AUC per fold (best):", auc_scores_rf)
print("Mean AUC:", auc_scores_rf.mean())
print("Std AUC:", auc_scores_rf.std())


print("Best RF AUC:", rf_search.best_score_)
print("Best RF params:", rf_search.best_params_)


Mean AUC (BASE): 0.8894008351996463
Best model stats: 
AUC per fold (best): [0.89529231 0.90363184 0.90533448 0.88973455 0.89431019]
Mean AUC: 0.8976606738343376
Std AUC: 0.005902549889825655
Best RF AUC: 0.8976606654405584
Best RF params: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': None}


# 3c - Based on chosen hyperparams, which is the best, using hold-out test set

**XGBoost** 
- Best XGBoost AUC: 0.9022729279222063
- Best XGBoost params: {'subsample': 1.0, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
- NEW'subsample': 0.8, 'n_estimators': 400, 'min_child_weight': 3, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 0.6}

**Random Forest**
- Best RF AUC: 0.8999472791583448
- Best RF params - {'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 30}
- Best RF params (OLD): {'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 30, 'bootstrap': True}
- (2, adding None params increasing runtime): {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': None}



# 4 - Measuring performance on the test set using best configurations

In [ ]:
# train our best hyperparams on all the training data

xgb_clf = XGBClassifier(
    subsample=1.0,
    n_estimators=400,
    min_child_weight=5,
    max_depth=8,
    learning_rate=0.1,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1   
)

rf_clf = RandomForestClassifier(
    n_jobs=-1,
    n_estimators=400,
    min_samples_split=2,
    min_samples_leaf=4,
    max_features="sqrt",
    max_depth=30   
)

xgb_clf.fit(X_train, y_train)
rf_clf.fit(X_train, y_train)

,n_estimators,400
,criterion,'gini'
,max_depth,30
,min_samples_split,2
,min_samples_leaf,4
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [14]:
# XGBoost test performance
best_xgb = xgb_clf
# best_xgb = xgb_search.best_estimator_
y_pred_xgb = best_xgb.predict_proba(X_test)[:, 1]
xgb_test_auc = roc_auc_score(y_test, y_pred_xgb)

print(f"\nXGBoost Test AUC: {xgb_test_auc:.4f}")

# Random Forest test performance
best_rf = rf_clf
# best_rf = rf_search.best_estimator_
y_pred_rf = best_rf.predict_proba(X_test)[:, 1]
rf_test_auc = roc_auc_score(y_test, y_pred_rf)

print(f"Random Forest Test AUC: {rf_test_auc:.4f}")

# Baseline test performance
baseline_clf = DummyClassifier(strategy='prior', random_state=seed)
baseline_clf.fit(X_train, y_train)
y_pred_baseline = baseline_clf.predict_proba(X_test)[:, 1]
baseline_test_auc = roc_auc_score(y_test, y_pred_baseline)

print(f"Baseline Test AUC: {baseline_test_auc:.4f}")


XGBoost Test AUC: 0.9019
Random Forest Test AUC: 0.8987
Baseline Test AUC: 0.5000


In [15]:
y_pred_rf

array([0.0910459 , 0.05432218, 0.8774887 , ..., 0.01636979, 0.02508396,
       0.09245053], shape=(40579,))

### Conclusion: XGboost AUC was slightly higher than Random forest, and both are much better than baseline model's
# 5 - next, estimate model performance with a confidence interval
- assumes normality

In [17]:
np.unique(y_test)

array([0., 1.])

In [23]:
def samples_for_t_ci(y_true, y_pred_proba, n_samples=100):
    np.random.seed(seed)
    n_all= len(y_true)
    sample_aucs = []
    confidence=0.95
    
    for i in range(n_samples):
        print(f"Progress: {(i/n_samples) * 100}%")
        # Sample with replacement
        indices = np.random.choice(n_all, size=n_samples, replace=True)
        
        auc = roc_auc_score(y_true[indices], y_pred_proba[indices])
        sample_aucs.append(auc)
    
    sample_aucs = np.array(sample_aucs)
    print(f"We have {len(sample_aucs)} auc scores")
    
    # other stats
    n = len(sample_aucs)
    m = np.mean(sample_aucs)
    std_dev = np.std(sample_aucs, ddof=1)
    
    # T-distribution CI based on slides
    alpha = 1 - confidence
    df = n - 1
    t_value = stats.t.ppf(1 - alpha/2, df)
    
    margin_of_err = t_value * (std_dev / np.sqrt(n))
    print(f"margin of error is: {margin_of_err}")
    ci_lower = m - margin_of_err
    ci_upper = m + margin_of_err
    
    return m, ci_lower, ci_upper, sample_aucs, t_value, std_dev, n

# Get predictions on test set for best model
# y_pred_proba = best_rf.predict_proba(X_test)[:, 1]
y_pred_proba = best_xgb.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)

# Calculate bootstrap-based t-distribution CI
best_mean, ci_lower, ci_upper, scores, t_value, std_dev, n_val = samples_for_t_ci(y_test.values, y_pred_proba, n_samples=100)


Progress: 0.0%
Progress: 1.0%
Progress: 2.0%
Progress: 3.0%
Progress: 4.0%
Progress: 5.0%
Progress: 6.0%
Progress: 7.000000000000001%
Progress: 8.0%
Progress: 9.0%
Progress: 10.0%
Progress: 11.0%
Progress: 12.0%
Progress: 13.0%
Progress: 14.000000000000002%
Progress: 15.0%
Progress: 16.0%
Progress: 17.0%
Progress: 18.0%
Progress: 19.0%
Progress: 20.0%
Progress: 21.0%
Progress: 22.0%
Progress: 23.0%
Progress: 24.0%
Progress: 25.0%
Progress: 26.0%
Progress: 27.0%
Progress: 28.000000000000004%
Progress: 28.999999999999996%
Progress: 30.0%
Progress: 31.0%
Progress: 32.0%
Progress: 33.0%
Progress: 34.0%
Progress: 35.0%
Progress: 36.0%
Progress: 37.0%
Progress: 38.0%
Progress: 39.0%
Progress: 40.0%
Progress: 41.0%
Progress: 42.0%
Progress: 43.0%
Progress: 44.0%
Progress: 45.0%
Progress: 46.0%
Progress: 47.0%
Progress: 48.0%
Progress: 49.0%
Progress: 50.0%
Progress: 51.0%
Progress: 52.0%
Progress: 53.0%
Progress: 54.0%
Progress: 55.00000000000001%
Progress: 56.00000000000001%
Progress: 56.999

In [29]:
print(f"Test set AUC: {test_auc:.4f}")
print(f"Number of bootstrap samples (n): {n_val}")
print(f"Samples mean: {best_mean:.4f}")
print(f"Samples std dev: {std_dev:.4f}")
print(f"t-value: {t_value:.4f}")
print(f"\n95% Confidence Interval: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("\n" + "*"*10)
print("Compared to Baseline")

baseline_clf = DummyClassifier(strategy='prior', random_state=seed)
baseline_clf.fit(X_train, y_train)
y_pred_baseline = baseline_clf.predict_proba(X_test)[:, 1]
baseline_test_auc = roc_auc_score(y_test, y_pred_baseline)

print(f"Best AUC: {test_auc:.4f}")
print(f"RF 95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"Baseline Test AUC: {baseline_test_auc:.4f}")

## check if Baseline AUC falls outside outside of best model CI, then we can say it outperforms the baseline!


Test set AUC: 0.9019
Number of bootstrap samples (n): 100
Samples mean: 0.8976
Samples std dev: 0.0764
t-value: 1.9842

95% Confidence Interval: [0.8825, 0.9128]

**********
Compared to Baseline
Best AUC: 0.9019
RF 95% CI: [0.8825, 0.9128]
Baseline Test AUC: 0.5000


## next notebook: train on the whole dataset with the given parameters